# 02 — Build Silver Retail Sales Clean Layer

Project: Databricks SQL Retail Analytics Dashboard  
Layer: Silver  
Source Table: retail_analytics_project.bronze_online_retail_raw  
Target Table: retail_analytics_project.silver_retail_sales_clean  

This notebook cleans and standardizes the raw Online Retail transactions stored in Bronze.

The Silver layer:

- standardizes column names;
- cleans text fields;
- handles missing customer and product values;
- classifies regular transactions and cancellations;
- creates data quality flags;
- calculates transactional amounts;
- adds date attributes for analytical consumption.

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    when,
    lit,
    coalesce,
    to_date,
    year,
    month,
    dayofmonth,
    date_format,
    current_timestamp,
    abs as spark_abs
)

In [0]:
schema_name = "retail_analytics_project"

bronze_table = f"{schema_name}.bronze_online_retail_raw"
silver_table = f"{schema_name}.silver_retail_sales_clean"

print(f"Bronze source: {bronze_table}")
print(f"Silver target: {silver_table}")

In [0]:
if not spark.catalog.tableExists(bronze_table):
    raise Exception(f"Required Bronze table does not exist: {bronze_table}")

print(f"Bronze table found: {bronze_table}")

In [0]:
bronze_df = spark.table(bronze_table)

bronze_count = bronze_df.count()

print(f"Bronze records available: {bronze_count}")

display(bronze_df.limit(10))

In [0]:
silver_base_df = (
    bronze_df
    .select(
        trim(col("InvoiceNo")).alias("invoice_no"),
        trim(col("StockCode")).alias("stock_code"),
        trim(col("Description")).alias("product_description_raw"),

        col("Quantity")
            .cast("int")
            .alias("quantity"),

        col("InvoiceDate")
            .cast("timestamp")
            .alias("invoice_ts"),

        col("UnitPrice")
            .cast("double")
            .alias("unit_price"),

        trim(col("CustomerID")).alias("customer_id_raw"),
        trim(col("Country")).alias("country_raw"),

        col("source_system"),
        col("source_file"),
        col("source_url"),
        col("bronze_ingestion_ts")
    )
)

In [0]:
silver_clean_df = (
    silver_base_df
    .withColumn(
        "customer_id",
        coalesce(col("customer_id_raw"), lit("UNKNOWN"))
    )
    .withColumn(
        "product_description",
        when(
            col("product_description_raw").isNull()
            | (trim(col("product_description_raw")) == ""),
            lit("UNKNOWN PRODUCT")
        ).otherwise(
            trim(col("product_description_raw"))
        )
    )
    .withColumn(
        "country",
        when(
            col("country_raw").isNull()
            | (trim(col("country_raw")) == ""),
            lit("UNKNOWN COUNTRY")
        ).otherwise(
            trim(col("country_raw"))
        )
    )
    
)

In [0]:
silver_classified_df = (
    silver_clean_df
    .withColumn(
        "is_customer_known",
        col("customer_id") != "UNKNOWN"
    )
    .withColumn(
        "is_cancellation",
        col("invoice_no").startswith("C")
    )
    .withColumn(
        "has_positive_quantity",
        col("quantity") > 0
    )
    .withColumn(
        "has_positive_unit_price",
        col("unit_price") > 0
    )
    .withColumn(
        "is_product_description_known",
        col("product_description") !=  "UNKNOWN PRODUCT"
    )
)

In [0]:
silver_classified_df = (
    silver_classified_df
    .withColumn(
        "transaction_type",
        when(col("is_cancellation"), lit("CANCELLATION"))
        .when(col("quantity") < 0, lit("RETURN_OR_ADJUSTMENT"))
        .when(
            (col("quantity") > 0) & (col("unit_price") > 0),
            lit("SALE")
        )
        .otherwise(lit("INVALID_OR_NON_COMMERCIAL"))
    )
)

In [0]:
display(
    silver_classified_df
    .groupBy("transaction_type")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
silver_classified_df = (
    silver_classified_df
    .withColumn(
        "is_valid_sales_line",
        (
            (col("transaction_type") == "SALE")
            & col("is_product_description_known")
            & col("stock_code").isNotNull()
        )
    )
)

In [0]:
from pyspark.sql.functions import round as spark_round

silver_amounts_df = (
    silver_classified_df
    .withColumn(
        "signed_line_amount",
        spark_round(
            col("quantity") * col("unit_price"),
            2
        )
    )
    .withColumn(
        "gross_sales_amount",
        when(
            col("is_valid_sales_line"),
            spark_round(
                col("quantity") * col("unit_price"),
                2
            )
        ).otherwise(lit(0.0))
    )
    .withColumn(
        "cancellation_amount",
        when(
            col("is_cancellation"),
            spark_round(
                spark_abs(
                    col("quantity") * col("unit_price")
                ),
                2
            )
        ).otherwise(lit(0.0))
    )
)

In [0]:
silver_final_df = (
    silver_amounts_df
    .withColumn("invoice_date", to_date(col("invoice_ts")))
    .withColumn("invoice_year", year(col("invoice_ts")))
    .withColumn("invoice_month", month(col("invoice_ts")))
    .withColumn("invoice_day", dayofmonth(col("invoice_ts")))
    .withColumn(
        "invoice_year_month",
        date_format(col("invoice_ts"), "yyyy-MM")
    )
    .withColumn(
        "silver_processed_ts",
        current_timestamp()
    )
)

In [0]:
silver_final_df = silver_final_df.select(
    "invoice_no",
    "stock_code",
    "product_description",
    "quantity",
    "invoice_ts",
    "invoice_date",
    "invoice_year",
    "invoice_month",
    "invoice_day",
    "invoice_year_month",
    "unit_price",
    "customer_id",
    "country",
    "transaction_type",
    "is_customer_known",
    "is_cancellation",
    "has_positive_quantity",
    "has_positive_unit_price",
    "is_product_description_known",
    "is_valid_sales_line",
    "signed_line_amount",
    "gross_sales_amount",
    "cancellation_amount",
    "source_system",
    "source_file",
    "source_url",
    "bronze_ingestion_ts",
    "silver_processed_ts"
)

display(silver_final_df.limit(20))

In [0]:
silver_final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_table)

print(f"Silver table created successfully: {silver_table}")

In [0]:
silver_count = spark.table(silver_table).count()

print(f"Silver record count: {silver_count}")

In [0]:
display(
    spark.sql(f"""
    SELECT
        transaction_type,
        COUNT(*) AS total_rows,
        ROUND(SUM(signed_line_amount), 2) AS signed_amount,
        ROUND(SUM(gross_sales_amount), 2) AS gross_sales_amount,
        ROUND(SUM(cancellation_amount), 2) AS cancellation_amount
    FROM {silver_table}
    GROUP BY transaction_type
    ORDER BY total_rows DESC
    """)
)

In [0]:
display(
    spark.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN customer_id = 'UNKNOWN' THEN 1 ELSE 0 END) AS unknown_customer_rows,
        SUM(CASE WHEN product_description = 'UNKNOWN PRODUCT' THEN 1 ELSE 0 END) AS unknown_product_rows,
        SUM(CASE WHEN is_valid_sales_line THEN 1 ELSE 0 END) AS valid_sales_rows,
        SUM(CASE WHEN is_cancellation THEN 1 ELSE 0 END) AS cancellation_rows,
        SUM(CASE WHEN NOT has_positive_unit_price THEN 1 ELSE 0 END) AS non_positive_price_rows,
        SUM(CASE WHEN NOT has_positive_quantity THEN 1 ELSE 0 END) AS non_positive_quantity_rows
    FROM {silver_table}
    """)
)

In [0]:
display(
    spark.sql(f"""
    SELECT
        invoice_no,
        stock_code,
        product_description,
        quantity,
        unit_price,
        gross_sales_amount,
        customer_id,
        country,
        invoice_ts
    FROM {silver_table}
    WHERE is_valid_sales_line = true
    ORDER BY invoice_ts
    LIMIT 20
    """)
)

In [0]:
display(
    spark.sql(f"""
    SELECT
        invoice_no,
        stock_code,
        product_description,
        quantity,
        unit_price,
        signed_line_amount,
        cancellation_amount,
        customer_id,
        country,
        invoice_ts
    FROM {silver_table}
    WHERE is_cancellation = true
    ORDER BY invoice_ts
    LIMIT 20
    """)
)

In [0]:
silver_count = spark.table(silver_table).count()

print(f"Silver record count: {silver_count}")

In [0]:
display(
    spark.sql(f"""
    SELECT
        transaction_type,
        COUNT(*) AS total_rows,
        ROUND(SUM(signed_line_amount), 2) AS signed_amount,
        ROUND(SUM(gross_sales_amount), 2) AS gross_sales_amount,
        ROUND(SUM(cancellation_amount), 2) AS cancellation_amount
    FROM {silver_table}
    GROUP BY transaction_type
    ORDER BY total_rows DESC
    """)
)